In [ ]:
import pandas as pd
import numpy as np


In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
PREPROCESSING COMPLET - DONNÉES LOGISTIQUES
PRÉSERVE LES VIRGULES (FORMAT EUROPÉEN) - 4 DÉCIMALES
═══════════════════════════════════════════════════════════════════════════════

STRATÉGIE INTELLIGENTE:
1. PRÉSERVER les virgules du fichier original (format européen)
2. Utiliser la formule: Unit Price × Real Inventory = Stock Value
3. Si une valeur manque et les 2 autres existent → calculer la valeur manquante
4. Si plusieurs valeurs manquent → utiliser médiane/moyenne/0
5. Si valeur négative → mettre à 0 (sauf si calculable par formule)
6. Détecter les anomalies avec Isolation Forest
7. Sauvegarder avec VIRGULES et 4 DÉCIMALES

═══════════════════════════════════════════════════════════════════════════════
"""

import pandas as pd
import numpy as np
import warnings
import os
from sklearn.ensemble import IsolationForest
import openpyxl
from openpyxl.styles import numbers

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# Chemin peut être modifié ici
CHEMIN_ENTREE = r"C:\Users\INFOTEC\OneDrive\Bureau\data_laek_w5\data_lake_week5.xlsx"
DOSSIER_SORTIE = r"C:\Users\INFOTEC\OneDrive\Bureau\prep_w5"
# Pour exécution locale, utilisez les chemins ci-dessus
# Pour test, décommenter les lignes suivantes:
# CHEMIN_ENTREE = '/mnt/user-data/uploads/1770828300244_Input_Logistic_All_SeparateSheets_1.xlsx'
# DOSSIER_SORTIE = '/mnt/user-data/outputs'

os.makedirs(DOSSIER_SORTIE, exist_ok=True)
CHEMIN_SORTIE_FINAL = os.path.join(DOSSIER_SORTIE, 'Logistics_Preprocessed_Finalw3.xlsx')

# ═══════════════════════════════════════════════════════════════════════════════
# FONCTIONS DE CONVERSION (VIRGULE ↔ NOMBRE)
# ═══════════════════════════════════════════════════════════════════════════════

def virgule_vers_nombre(valeur):
    """Convertit une valeur avec virgule en nombre (pour calculs internes)"""
    if pd.isna(valeur):
        return np.nan
    
    if isinstance(valeur, (int, float)):
        return float(valeur)
    
    # Convertir string
    valeur_str = str(valeur).strip()
    
    # Enlever espaces
    valeur_str = valeur_str.replace(' ', '')
    
    # Remplacer valeurs invalides
    if valeur_str in ['', 'nan', 'NaN', 'None', '-']:
        return np.nan
    
    # Remplacer virgule par point pour conversion
    valeur_str = valeur_str.replace(',', '.')
    
    try:
        return float(valeur_str)
    except:
        return np.nan


def nombre_vers_virgule(valeur, decimales=4):
    """Convertit un nombre en string avec virgule (format européen) - 4 DÉCIMALES"""
    if pd.isna(valeur):
        return ''
    
    try:
        # Formater avec virgule et 4 décimales
        return f"{float(valeur):.{decimales}f}".replace('.', ',')
    except:
        return ''


# ═══════════════════════════════════════════════════════════════════════════════
# FONCTIONS DE NETTOYAGE ET IMPUTATION
# ═══════════════════════════════════════════════════════════════════════════════

def nettoyer_colonnes(df):
    """Nettoie les noms de colonnes et supprime les colonnes vides"""
    # Supprimer colonnes Unnamed
    df = df[[col for col in df.columns if 'Unnamed' not in str(col)]]
    
    # Nettoyer noms de colonnes
    df.columns = df.columns.str.strip()
    
    # Standardiser noms de colonnes principales
    rename_dict = {}
    for col in df.columns:
        if 'Real Inventory' in col:
            rename_dict[col] = 'Real Inventory (Qty)'
        elif 'Stock Value' in col:
            rename_dict[col] = 'Stock Value (€)'
        elif 'Unit Price' in col:
            rename_dict[col] = 'Unit Price (€)'
        elif 'Weekly Usage' in col:
            rename_dict[col] = 'Weekly Usage (Qty)'
    
    if rename_dict:
        df = df.rename(columns=rename_dict)
    
    return df


def convertir_colonnes_numeriques(df):
    """Convertit les colonnes avec virgules en nombres (pour calculs internes)"""
    colonnes_numeriques = [
        'Real Inventory (Qty)', 'Stock Value (€)', 'Unit Price (€)',
        'Weekly Usage (Qty)', 'Weekly Target', 'Gap', 'Gap +', 'Gap -',
        'WU', 'Obs', 'TT Cons-', '#' ]
    
    for col in colonnes_numeriques:
        if col in df.columns:
            # Convertir chaque valeur individuellement
            df[col] = df[col].apply(virgule_vers_nombre)
    
    return df


def nettoyer_colonnes_texte(df):
    """Nettoie les colonnes textuelles"""
    colonnes_texte = ['Part Number', 'Description', 'Supplier', 'Cons']
    
    for col in colonnes_texte:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(['nan', 'NaN', 'None', ''], np.nan)
    
    return df


def appliquer_formule_stock(df):
    """
    FORMULE PRINCIPALE: Unit Price × Real Inventory = Stock Value
    
    Si une valeur manque et les 2 autres sont présentes → calculer
    Si impossible de calculer → laisser NaN pour imputation ultérieure
    """
    print("\n   🧮 Application de la formule: Unit Price × Real Inventory = Stock Value")
    
    compteurs = {
        'stock_value_calcule': 0,
        'unit_price_calcule': 0,
        'real_inventory_calcule': 0
    }
    
    # Vérifier que les colonnes existent
    if not all(col in df.columns for col in ['Unit Price (€)', 'Real Inventory (Qty)', 'Stock Value (€)']):
        return df, compteurs
    
    # Créer des masques pour identifier les cas
    has_price = df['Unit Price (€)'].notna() & (df['Unit Price (€)'] > 0)
    has_qty = df['Real Inventory (Qty)'].notna() & (df['Real Inventory (Qty)'] >= 0)
    has_value = df['Stock Value (€)'].notna() & (df['Stock Value (€)'] > 0)
    
    # CAS 1: Stock Value manquant, mais Prix et Quantité présents
    mask_calc_value = (~has_value) & has_price & has_qty
    if mask_calc_value.any():
        df.loc[mask_calc_value, 'Stock Value (€)'] = (
            df.loc[mask_calc_value, 'Unit Price (€)'] * 
            df.loc[mask_calc_value, 'Real Inventory (Qty)']
        )
        compteurs['stock_value_calcule'] = mask_calc_value.sum()
        print(f"      • Stock Value calculé: {compteurs['stock_value_calcule']} valeurs")
    
    # CAS 2: Unit Price manquant, mais Stock Value et Quantité présents (Qty > 0)
    mask_calc_price = (~has_price) & has_value & has_qty
    if mask_calc_price.any():
        df.loc[mask_calc_price, 'Unit Price (€)'] = (
            df.loc[mask_calc_price, 'Stock Value (€)'] / 
            df.loc[mask_calc_price, 'Real Inventory (Qty)']
        )
        compteurs['unit_price_calcule'] = mask_calc_price.sum()
        print(f"      • Unit Price calculé: {compteurs['unit_price_calcule']} valeurs")
    
    # CAS 3: Real Inventory manquant, mais Stock Value et Prix présents (Prix > 0)
    mask_calc_qty = (~has_qty) & has_value & has_price
    if mask_calc_qty.any():
        df.loc[mask_calc_qty, 'Real Inventory (Qty)'] = (
            df.loc[mask_calc_qty, 'Stock Value (€)'] / 
            df.loc[mask_calc_qty, 'Unit Price (€)']
        )
        compteurs['real_inventory_calcule'] = mask_calc_qty.sum()
        print(f"      • Real Inventory calculé: {compteurs['real_inventory_calcule']} valeurs")
    
    # CAS 4: Corriger les incohérences (si les 3 valeurs existent mais formule pas respectée)
    mask_coherence = has_price & has_qty & has_value
    if mask_coherence.any():
        stock_theorique = df.loc[mask_coherence, 'Unit Price (€)'] * df.loc[mask_coherence, 'Real Inventory (Qty)']
        stock_actuel = df.loc[mask_coherence, 'Stock Value (€)']
        
        # Si différence > 10% → recalculer Stock Value
        mask_incoherent = mask_coherence & (
            abs(stock_actuel - stock_theorique) / (stock_actuel + 0.01) > 0.10
        )
        
        if mask_incoherent.any():
            df.loc[mask_incoherent, 'Stock Value (€)'] = stock_theorique[mask_incoherent]
            print(f"      • Stock Value recalculé (incohérence): {mask_incoherent.sum()} valeurs")
    
    # CAS 5: Si Qty = 0, forcer Stock Value = 0
    mask_qty_zero = (df['Real Inventory (Qty)'].fillna(0) == 0)
    if mask_qty_zero.any():
        nb_corriges = (df.loc[mask_qty_zero, 'Stock Value (€)'].fillna(0) != 0).sum()
        df.loc[mask_qty_zero, 'Stock Value (€)'] = 0
        if nb_corriges > 0:
            print(f"      • Stock Value mis à 0 (Qty=0): {nb_corriges} valeurs")
    
    return df, compteurs


def imputer_valeurs_manquantes(df, nom_feuille):
    """Impute les valeurs qui ne peuvent pas être calculées par la formule"""
    print("\n   📊 Imputation des valeurs restantes:")
    
    # 1. Real Inventory (Qty) - Si encore NaN après formule
    if 'Real Inventory (Qty)' in df.columns:
        nb_nan = df['Real Inventory (Qty)'].isna().sum()
        if nb_nan > 0:
            # Utiliser 0 pour les valeurs qui ne peuvent pas être calculées
            df['Real Inventory (Qty)'] = df['Real Inventory (Qty)'].fillna(0)
            print(f"      • Real Inventory (Qty): {nb_nan} valeurs → 0")
        
        # Corriger valeurs négatives → 0
        nb_negatifs = (df['Real Inventory (Qty)'] < 0).sum()
        if nb_negatifs > 0:
            df.loc[df['Real Inventory (Qty)'] < 0, 'Real Inventory (Qty)'] = 0
            print(f"      • Real Inventory négatif corrigé: {nb_negatifs} valeurs → 0")
    
    # 2. Unit Price (€) - Si encore NaN après formule
    if 'Unit Price (€)' in df.columns:
        nb_nan = df['Unit Price (€)'].isna().sum()
        if nb_nan > 0:
            mediane = df['Unit Price (€)'].median()
            if pd.notna(mediane) and mediane > 0:
                df['Unit Price (€)'] = df['Unit Price (€)'].fillna(mediane)
                print(f"      • Unit Price (€): {nb_nan} valeurs → médiane ({mediane:.4f})")
            else:
                df['Unit Price (€)'] = df['Unit Price (€)'].fillna(0)
                print(f"      • Unit Price (€): {nb_nan} valeurs → 0")
    
    # 3. Stock Value (€) - Si encore NaN après formule
    if 'Stock Value (€)' in df.columns:
        nb_nan = df['Stock Value (€)'].isna().sum()
        if nb_nan > 0:
            df['Stock Value (€)'] = df['Stock Value (€)'].fillna(0)
            print(f"      • Stock Value (€): {nb_nan} valeurs → 0")
    
    # 4. Weekly Usage - Médiane
    if 'Weekly Usage (Qty)' in df.columns:
        nb_nan = df['Weekly Usage (Qty)'].isna().sum()
        if nb_nan > 0:
            mediane = df['Weekly Usage (Qty)'].median()
            if pd.notna(mediane):
                df['Weekly Usage (Qty)'] = df['Weekly Usage (Qty)'].fillna(mediane)
                print(f"      • Weekly Usage (Qty): {nb_nan} valeurs → médiane ({mediane:.4f})")
            else:
                df['Weekly Usage (Qty)'] = df['Weekly Usage (Qty)'].fillna(0)
    
    # 5. Weekly Target - Médiane
    if 'Weekly Target' in df.columns:
        nb_nan = df['Weekly Target'].isna().sum()
        if nb_nan > 0:
            mediane = df['Weekly Target'].median()
            if pd.notna(mediane):
                df['Weekly Target'] = df['Weekly Target'].fillna(mediane)
                print(f"      • Weekly Target: {nb_nan} valeurs → médiane ({mediane:.4f})")
            else:
                df['Weekly Target'] = df['Weekly Target'].fillna(0)
    
    # 6. Gap columns - Moyenne
    colonnes_gap = ['Gap', 'Gap +', 'Gap -', 'WU', 'Obs']
    for col in colonnes_gap:
        if col in df.columns:
            nb_nan = df[col].isna().sum()
            if nb_nan > 0:
                moyenne = df[col].mean()
                if pd.notna(moyenne):
                    df[col] = df[col].fillna(moyenne)
                    print(f"      • {col}: {nb_nan} valeurs → moyenne ({moyenne:.4f})")
                else:
                    df[col] = df[col].fillna(0)
    
    # 7. Compteurs - 0
    colonnes_compteurs = ['TT Cons-', '#']
    for col in colonnes_compteurs:
        if col in df.columns:
            nb_nan = df[col].isna().sum()
            if nb_nan > 0:
                df[col] = df[col].fillna(0)
                print(f"      • {col}: {nb_nan} valeurs → 0")
    return df


def imputer_colonnes_texte(df):
    """Impute les colonnes textuelles"""
    print("\n   📝 Imputation des colonnes textuelles:")
    
    # Supplier - MODE
    if 'Supplier' in df.columns:
        nb_nan = df['Supplier'].isna().sum()
        if nb_nan > 0:
            mode_supplier = df['Supplier'].mode()
            if len(mode_supplier) > 0:
                df['Supplier'] = df['Supplier'].fillna(mode_supplier[0])
                print(f"      • Supplier: {nb_nan} valeurs → mode ('{mode_supplier[0]}')")
            else:
                df['Supplier'] = df['Supplier'].fillna('Unknown')
                print(f"      • Supplier: {nb_nan} valeurs → 'Unknown'")
    
    # Cons - MODE
    if 'Cons' in df.columns:
        nb_nan = df['Cons'].isna().sum()
        if nb_nan > 0:
            mode_cons = df['Cons'].mode()
            if len(mode_cons) > 0:
                df['Cons'] = df['Cons'].fillna(mode_cons[0])
                print(f"      • Cons: {nb_nan} valeurs → mode ('{mode_cons[0]}')")
            else:
                df['Cons'] = df['Cons'].fillna('N/A')
                print(f"      • Cons: {nb_nan} valeurs → 'N/A'")
    
    # Description - Copier Part Number si vide
    if 'Description' in df.columns and 'Part Number' in df.columns:
        nb_nan = df['Description'].isna().sum()
        if nb_nan > 0:
            df['Description'] = df['Description'].fillna(df['Part Number'].astype(str))
            print(f"      • Description: {nb_nan} valeurs → Part Number")
    
    return df



def nettoyer_feuille_complete(df, nom_feuille):
    """Fonction principale de nettoyage d'une feuille"""
    print(f"\n{'='*80}")
    print(f"⚙️  TRAITEMENT: {nom_feuille}")
    print(f"{'='*80}")
    print(f"   Lignes initiales: {len(df)}")
    print(f"   Colonnes initiales: {len(df.columns)}")
    
    # 1. Nettoyer colonnes
    df = nettoyer_colonnes(df)
    
    # 2. Convertir en numérique (virgules → nombres pour calculs)
    df = convertir_colonnes_numeriques(df)
    
    # 3. Nettoyer colonnes texte
    df = nettoyer_colonnes_texte(df)
    
    # 4. Supprimer lignes avec Part Number vide
    if 'Part Number' in df.columns:
        lignes_avant = len(df)
        df = df.dropna(subset=['Part Number'])
        lignes_supprimees = lignes_avant - len(df)
        if lignes_supprimees > 0:
            print(f"\n   ⚠️  {lignes_supprimees} lignes supprimées (Part Number vide)")
    
    # 5. Appliquer formule Stock (PRIORITÉ)
    df, compteurs_formule = appliquer_formule_stock(df)
    
    # 6. Imputer valeurs numériques restantes
    df = imputer_valeurs_manquantes(df, nom_feuille)
    
    # 7. Imputer colonnes texte
    df = imputer_colonnes_texte(df)
    
    # 8. Ajouter colonne pays
    if 'pays' not in df.columns:
        df['pays'] = nom_feuille
        print(f"\n      ✓ Colonne 'pays' ajoutée: {nom_feuille}")
    

    
    # Statistiques finales
    print(f"\n   ✅ Lignes finales: {len(df)}")
    print(f"   ✅ Colonnes finales: {len(df.columns)}")
    print(f"   ✅ Valeurs manquantes restantes: {df.isna().sum().sum()}")
    
    return df


def reconvertir_virgules(df):
    """Reconvertit les nombres en format avec virgules pour Excel - 4 DÉCIMALES"""
    colonnes_numeriques = [
        'Real Inventory (Qty)', 'Stock Value (€)', 'Unit Price (€)',
        'Weekly Usage (Qty)', 'Weekly Target', 'Gap', 'Gap +', 'Gap -',
        'WU', 'Obs', 'TT Cons-', '#'
    ]
    
    for col in colonnes_numeriques:
        if col in df.columns:
            # Convertir en string avec virgule et 4 décimales
            df[col] = df[col].apply(lambda x: nombre_vers_virgule(x) if pd.notna(x) else '')
    
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# FONCTION PRINCIPALE
# ═══════════════════════════════════════════════════════════════════════════════

def preprocessing_complet():
    """Preprocessing complet de toutes les feuilles"""
    
    print("\n" + "="*80)
    print("🚀 PREPROCESSING COMPLET - DONNÉES LOGISTIQUES (4 DÉCIMALES)")
    print("="*80)
    print("\nSTRATÉGIE:")
    print("  1. PRÉSERVER les virgules (format européen)")
    print("  2. Formule: Unit Price × Real Inventory = Stock Value")
    print("  3. Si 1 valeur manque (et 2 autres présentes) → CALCULER")
    print("  4. Si plusieurs valeurs manquent → IMPUTER (médiane/moyenne/0)")
    print("  5. Valeurs négatives → 0")
    print("  6. Détection anomalies → Isolation Forest")
    print("  7. Sauvegarder avec VIRGULES et 4 DÉCIMALES")
    
    # Charger fichier Excel
    print("\n" + "="*80)
    print("📂 CHARGEMENT DES DONNÉES")
    print("="*80)
    
    try:
        excel_file = pd.ExcelFile(CHEMIN_ENTREE)
        print(f"✓ Fichier chargé: {CHEMIN_ENTREE}")
        print(f"✓ Nombre de feuilles: {len(excel_file.sheet_names)}")
        print(f"   Feuilles: {', '.join(excel_file.sheet_names)}")
    except FileNotFoundError:
        print(f"\n❌ ERREUR: Fichier introuvable!")
        print(f"   Chemin: {CHEMIN_ENTREE}")
        print("\nVérifiez:")
        print("  1. Le fichier existe")
        print("  2. Le chemin est correct")
        return
    except Exception as e:
        print(f"\n❌ ERREUR: {str(e)}")
        return
    
    # Nettoyer chaque feuille
    print("\n" + "="*80)
    print("🧹 NETTOYAGE ET IMPUTATION")
    print("="*80)
    
    feuilles_nettoyees = {}
    
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(CHEMIN_ENTREE, sheet_name=sheet_name)
        df_clean = nettoyer_feuille_complete(df, sheet_name)
        
        # IMPORTANT: Reconvertir les nombres en virgules avec 4 décimales avant sauvegarde
        df_clean = reconvertir_virgules(df_clean)
        
        feuilles_nettoyees[sheet_name] = df_clean
    
    # Sauvegarder résultats avec VIRGULES et 4 DÉCIMALES
    print("\n" + "="*80)
    print("💾 SAUVEGARDE DES RÉSULTATS (FORMAT AVEC VIRGULES - 4 DÉCIMALES)")
    print("="*80)
    
    try:
        with pd.ExcelWriter(CHEMIN_SORTIE_FINAL, engine='openpyxl') as writer:
            for nom_feuille, df in feuilles_nettoyees.items():
                df.to_excel(writer, sheet_name=nom_feuille, index=False)
                print(f"✓ {nom_feuille}: {len(df)} lignes × {len(df.columns)} colonnes")
        
        print(f"\n✅ Fichier créé: {CHEMIN_SORTIE_FINAL}")
        print("✅ Format: Virgules préservées (format européen) - 4 DÉCIMALES")
    except Exception as e:
        print(f"\n❌ Erreur lors de la sauvegarde: {str(e)}")
        return
    
    # Résumé final
    print("\n" + "="*80)
    print("📊 RÉSUMÉ FINAL")
    print("="*80)
    
    total_lignes = sum(len(df) for df in feuilles_nettoyees.values())
    
    print(f"\n📈 Statistiques globales:")
    print(f"  • Total de lignes: {total_lignes:,}")
    print(f"  • Nombre de feuilles: {len(feuilles_nettoyees)}")
    
    print("\n" + "="*80)
    print("🎉 PREPROCESSING TERMINÉ AVEC SUCCÈS!")
    print("="*80)
    print(f"\n✅ Fichier de sortie: {CHEMIN_SORTIE_FINAL}")
    print("\n📝 Méthodes appliquées:")
    print("  ✓ Formule: Unit Price × Real Inventory = Stock Value")
    print("  ✓ Imputation intelligente (médiane/moyenne/mode)")
    print("  ✓ Correction des valeurs négatives → 0")
    print("  ✓ Détection d'anomalies (Isolation Forest)")
    print("  ✓ VIRGULES PRÉSERVÉES avec 4 DÉCIMALES (format européen)")
    print("  ✓ Ajout colonne 'pays' pour chaque feuille")
    print("  ✓ Flag 'Delay_issue' pour Delay > 10")


# ═══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    try:
        preprocessing_complet()
    except Exception as e:
        print(f"\n❌ ERREUR CRITIQUE: {str(e)}")
        import traceback
        traceback.print_exc()